In [ ]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Procore — Requisition Line Items (Bronze)
# MAGIC Pulls all Requisitions across all projects, then pulls Contract Detail
# MAGIC Line Items for each requisition and saves to the Bronze lakehouse.
 
# COMMAND ----------
 
import requests
import json
import time
import datetime
import pandas as pd
import re
import collections
 
# COMMAND ----------
 
# MAGIC %md
# MAGIC ## 1. Auth
 
# COMMAND ----------
 
auth = json.loads(mssparkutils.notebook.run("procore_auth", 90))
token = auth["token"]
COMPANY_ID = auth["company_id"]
 
headers = {
    "Authorization": f"Bearer {token}",
    "Procore-Company-Id": str(COMPANY_ID)
}
 
print("Auth successful" if token else "Auth failed")

StatementMeta(, c143d3b5-c056-43f5-a651-c2b9921af87d, 3, Finished, Available, Finished, False)

Auth successful


In [2]:
# COMMAND ----------
 
# MAGIC %md
# MAGIC ## 2. Get all projects
 
# COMMAND ----------
 
projects_resp = requests.get(
    "https://api.procore.com/rest/v1.0/projects",
    headers=headers,
    params={"company_id": COMPANY_ID, "per_page": 200}
)
projects_resp.raise_for_status()
projects = projects_resp.json()
print(f"{len(projects)} project(s) found")

StatementMeta(, c143d3b5-c056-43f5-a651-c2b9921af87d, 4, Finished, Available, Finished, False)

18 project(s) found


In [3]:
# COMMAND ----------
 
# MAGIC %md
# MAGIC ## 3. Pull all requisitions across all projects
 
# COMMAND ----------
 
all_requisitions = []
 
for project in projects:
    project_id = project["id"]
    project_name = project["name"]
 
    page = 1
    while True:
        resp = requests.get(
            "https://api.procore.com/rest/v1.1/requisitions",
            headers=headers,
            params={"project_id": project_id, "page": page, "per_page": 100}
        )
 
        if resp.status_code == 429:
            print(f"  Rate limited — waiting 30s...")
            time.sleep(30)
            continue
 
        if resp.status_code != 200:
            break
 
        rows = resp.json()
 
        if not rows or isinstance(rows, dict):
            break
 
        for row in rows:
            row["project_id"] = project_id
            row["project_name"] = project_name
 
        all_requisitions.extend(rows)
 
        if len(rows) < 100:
            break
 
        page += 1
        time.sleep(0.3)
 
print(f"Total requisitions found: {len(all_requisitions)}")
 
# Status breakdown
statuses = collections.Counter([r.get("status", "unknown") for r in all_requisitions])
print("\nStatus breakdown:")
for status, count in statuses.items():
    print(f"  {status}: {count}")

StatementMeta(, c143d3b5-c056-43f5-a651-c2b9921af87d, 5, Finished, Available, Finished, False)

Total requisitions found: 658

Status breakdown:
  approved: 393
  under_review: 115
  draft: 16
  pending_owner_approval: 116
  revise_and_resubmit: 3
  approved_as_noted: 15


In [4]:
# COMMAND ----------
 
# MAGIC %md
# MAGIC ## 4. Pull contract detail line items for each requisition
 
# COMMAND ----------
 
all_line_items = []
skipped = 0
 
print(f"Pulling line items for {len(all_requisitions)} requisition(s)...")
 
for req in all_requisitions:
    req_id = req["id"]
    project_id = req["project_id"]
 
    max_retries = 3
    retry_count = 0
 
    while retry_count < max_retries:
        resp = requests.get(
            f"https://api.procore.com/rest/v1.0/requisitions/{req_id}/contract_detail_items",
            headers=headers,
            params={"project_id": project_id}
        )
 
        if resp.status_code == 429:
            print(f"  Rate limited — waiting 30s...")
            time.sleep(30)
            retry_count += 1
            continue
 
        if resp.status_code != 200:
            skipped += 1
            break
 
        rows = resp.json()
 
        if not rows or isinstance(rows, dict):
            break
 
        for row in rows:
            row["requisition_id"] = req_id
            row["requisition_number"] = req.get("number", "")
            row["requisition_status"] = req.get("status", "")
            row["vendor_id"] = req.get("vendor_id", "")
            row["vendor_name"] = req.get("vendor_name", "")
            row["commitment_id"] = req.get("commitment_id", "")
            row["billing_date"] = req.get("billing_date", "")
            row["invoice_number"] = req.get("invoice_number", "")
            row["total_claimed_amount"] = req.get("total_claimed_amount", "")
            row["amount_due"] = req.get("payment_summary", {}).get("invoiced_amount_due", "")
            row["invoice_paid_in_full"] = req.get("payment_summary", {}).get("invoice_paid_in_full", False)
            row["project_id"] = project_id
            row["project_name"] = req.get("project_name", "")
 
        all_line_items.extend(rows)
        break
 
    time.sleep(1)
 
print(f"\nTotal line items pulled: {len(all_line_items)}")
print(f"Requisitions skipped (no data/error): {skipped}")

StatementMeta(, c143d3b5-c056-43f5-a651-c2b9921af87d, 6, Finished, Cancelled, Cancelled, False)

Pulling line items for 658 requisition(s)...
  Rate limited — waiting 30s...
  Rate limited — waiting 30s...
  Rate limited — waiting 30s...
  Rate limited — waiting 30s...
  Rate limited — waiting 30s...
  Rate limited — waiting 30s...
  Rate limited — waiting 30s...
  Rate limited — waiting 30s...
  Rate limited — waiting 30s...
  Rate limited — waiting 30s...
  Rate limited — waiting 30s...
  Rate limited — waiting 30s...
  Rate limited — waiting 30s...
  Rate limited — waiting 30s...
  Rate limited — waiting 30s...
  Rate limited — waiting 30s...
  Rate limited — waiting 30s...
  Rate limited — waiting 30s...
  Rate limited — waiting 30s...
  Rate limited — waiting 30s...


In [5]:
# COMMAND ----------
 
# MAGIC %md
# MAGIC ## 5. Save to Bronze lakehouse
 
# COMMAND ----------
 
BRONZE_TABLE = "procore_requisition_line_items_raw"
 
def clean_column_name(col):
    col = col.strip()
    col = re.sub(r'[ ,;{}()\n\t=]', '_', col)
    col = re.sub(r'_+', '_', col)
    col = col.strip('_')
    return col
 
if all_line_items:
    clean_rows = []
    for row in all_line_items:
        clean_row = {}
        for key, value in row.items():
            if value is None:
                clean_row[key] = None
            elif isinstance(value, (dict, list)):
                clean_row[key] = json.dumps(value)
            elif isinstance(value, bool):
                clean_row[key] = str(value)
            elif isinstance(value, (int, float, str)):
                clean_row[key] = value
            else:
                clean_row[key] = str(value)
        clean_rows.append(clean_row)
 
    pdf = pd.DataFrame(clean_rows)
    pdf.columns = [clean_column_name(c) for c in pdf.columns]
 
    for col in pdf.columns:
        if pdf[col].dtype == object:
            pdf[col] = pdf[col].astype(str).replace('None', None)
 
    spark.sql(f"DROP TABLE IF EXISTS {BRONZE_TABLE}")
    df = spark.createDataFrame(pdf)
    df.write.format("delta").mode("append").saveAsTable(BRONZE_TABLE)
    print(f"Wrote {df.count()} row(s) to {BRONZE_TABLE}")
else:
    print("No line items pulled this run — nothing written to bronze.")

StatementMeta(, c143d3b5-c056-43f5-a651-c2b9921af87d, 7, Finished, Available, Finished, False)

UnsupportedOperationException: No default context found, please attach a lakehouse before running spark sql queries with partial namespaces.